# Alibi CounterfactualProto — Interpretazione dei Risultati

Questo notebook documenta e interpreta le spiegazioni controfattuali generate da `CounterfactualProto` per due casi studio del modello XGBoost di previsione dell'abbandono.

**Cos'è un controfattuale?**  
Un controfattuale risponde alla domanda: *"Qual è la modifica minima al profilo di questo dipendente che invertirebbe la previsione del modello da Lascia → Rimane?"*  
Non è una prescrizione causale — rivela a quali feature il modello è più sensibile per questo specifico individuo.

---

## Configurazione

- Modello: XGBoost (200 alberi, max_depth=4, lr=0.05)
- Dati di addestramento: dataset IBM HR, split 80/20, bilanciato con SMOTE (1.176 campioni dopo il bilanciamento)
- Explainer: `CounterfactualProto` (alibi 0.9.6), modalità k-d tree, 500 iterazioni
- Classe target: 0 (Rimane)

---

## Caso 1 — Dipendente 214 (Vero Positivo)

**Previsione del modello:** 96,7% di probabilità di abbandono  
**Esito reale:** Left ✓ (il modello aveva ragione)

### Output controfattuale grezzo

P(Left): **96,7% → 40,2%**

| Feature | Originale | Controfattuale | Variazione |
|---|---|---|---|
| MonthlyRate | 9150,0 | 9172,266 | +22,266 |
| DailyRate | 756,0 | 733,735 | −22,265 |
| MonthlyIncome | 2174,0 | 2196,265 | +22,265 |
| HourlyRate | 99,0 | 84,588 | −14,412 |
| YearsWithCurrManager | 2,0 | 11,198 | +9,198 |
| TrainingTimesLastYear | 3,0 | 6,0 | +3,0 |
| JobInvolvement | 2,0 | 4,0 | +2,0 |
| RelationshipSatisfaction | 3,0 | 4,0 | +1,0 |
| BusinessTravel_Travel_Frequently | 1,0 | 0,0 | −1,0 |
| WorkLifeBalance | 3,0 | 4,0 | +1,0 |
| OverTime_Yes | 1,0 | 0,0 | −1,0 |
| NumCompaniesWorked | 1,0 | 0,0 | −1,0 |
| YearsAtCompany | 3,0 | 2,013 | −0,987 |
| PercentSalaryHike | 11,0 | 11,592 | +0,592 |
| YearsInCurrentRole | 2,0 | 1,408 | −0,592 |
| EnvironmentSatisfaction | 1,0 | 1,592 | +0,592 |
| Age | 21,0 | 21,592 | +0,592 |
| StockOptionLevel | 0,0 | 0,197 | +0,197 |
| JobSatisfaction | 2,0 | 1,803 | −0,197 |
| YearsSinceLastPromotion | 1,0 | 0,803 | −0,197 |
| DistanceFromHome | 1,0 | 1,197 | +0,197 |
| BusinessTravel_Travel_Rarely | 0,0 | 0,197 | +0,197 |
| Gender_Male | 0,0 | 0,197 | +0,197 |

### Interpretazione

Il controfattuale individua un insieme di cambiamenti legati al **carico di lavoro e al rapporto con il manager** come le leve più impattanti:

- **Eliminare gli straordinari** (`OverTime_Yes`: 1 → 0): l'intervento HR più immediatamente attuabile. Gli straordinari erano uno dei principali driver SHAP per questo dipendente.
- **Ridurre le trasferte frequenti** (`BusinessTravel_Travel_Frequently`: 1 → 0): rimozione di un fattore di stress cronico per un dipendente di livello base.
- **Aumentare la continuità con il manager** (`YearsWithCurrManager`: 2 → ~11): lo spostamento numerico più ampio. Riflette il forte peso che il modello assegna alla stabilità manageriale — questo dipendente aveva trascorso poco tempo con il manager attuale.
- **Aumentare il coinvolgimento lavorativo** (`JobInvolvement`: 2 → 4) e **la frequenza di formazione** (`TrainingTimesLastYear`: 3 → 6): entrambi segnalano la necessità di investire nell'engagement e nello sviluppo professionale.
- **Migliorare l'equilibrio vita-lavoro** (`WorkLifeBalance`: 3 → 4) e **la soddisfazione relazionale** (`RelationshipSatisfaction`: 3 → 4).
- La variazione del reddito (+€22) è trascurabile e probabilmente un artefatto dell'ottimizzatore — non un segnale HR significativo.

> **Conclusione per HR:** L'abbandono di questo dipendente è stato guidato da una combinazione di lavoro eccessivo, alto carico di trasferte e scarsa stabilità manageriale. Il controfattuale suggerisce che la rimozione degli straordinari e la riduzione delle trasferte, insieme a una maggiore continuità manageriale e a opportunità di sviluppo, sarebbero state le azioni di retention più efficaci.

---

## Caso 2 — Dipendente 223 (Falso Positivo)

**Previsione del modello:** 86,1% di probabilità di abbandono  
**Esito reale:** Stayed ✗ (il modello aveva torto)

### Output controfattuale grezzo

P(Lascia): **86,1% → 41,6%**

| Feature | Originale | Controfattuale | Variazione |
|---|---|---|---|
| MonthlyIncome | 1200,0 | 1254,374 | +54,374 |
| DailyRate | 812,0 | 861,744 | +49,744 |
| MonthlyRate | 9724,0 | 9698,046 | −25,954 |
| HourlyRate | 69,0 | 88,554 | +19,554 |
| Age | 18,0 | 29,732 | +11,732 |
| DistanceFromHome | 10,0 | 2,040 | −7,960 |
| TrainingTimesLastYear | 2,0 | 6,0 | +4,0 |
| EnvironmentSatisfaction | 4,0 | 1,0 | −3,0 |
| PercentSalaryHike | 12,0 | 14,467 | +2,467 |
| JobInvolvement | 2,0 | 4,0 | +2,0 |
| WorkLifeBalance | 3,0 | 1,539 | −1,461 |
| TotalWorkingYears | 0,0 | 1,066 | +1,066 |
| YearsAtCompany | 0,0 | 1,066 | +1,066 |
| YearsInCurrentRole | 0,0 | 1,066 | +1,066 |
| YearsSinceLastPromotion | 0,0 | 1,066 | +1,066 |
| YearsWithCurrManager | 0,0 | 1,066 | +1,066 |
| JobSatisfaction | 3,0 | 4,0 | +1,0 |
| EducationField_Marketing | 0,0 | 0,355 | +0,355 |
| MaritalStatus_Married | 0,0 | 0,355 | +0,355 |
| NumCompaniesWorked | 1,0 | 0,694 | −0,306 |

### Interpretazione

Questo è il caso analiticamente più interessante. Il modello prevedeva l'86,1% di probabilità di abbandono, eppure il dipendente è rimasto. Il controfattuale rivela **perché il modello ha sovrastimato il rischio** e quale profilo associa alla retention:

- **Età** (`Age`: 18 → ~30): il modello considera questo dipendente ad alto rischio semplicemente perché ha 18 anni. Il controfattuale afferma in sostanza *"se fosse più anziano, il modello sarebbe meno preoccupato"* — evidenziando un bias demografico verso la segnalazione dei lavoratori giovani/nuovi.
- **Feature legate all'anzianità** (`TotalWorkingYears`, `YearsAtCompany`, `YearsInCurrentRole`, `YearsWithCurrManager`, tutte 0 → ~1): il dipendente è appena assunto (0 anni in tutte le dimensioni di anzianità). Il controfattuale necessita anche solo di un anno di anzianità per ridurre il rischio previsto, evidenziando che il modello penalizza sproporzionatamente i nuovi assunti.
- **Distanza da casa** (`DistanceFromHome`: 10 → 2): una riduzione significativa. Il modello tratta la distanza dal luogo di lavoro come un fattore di rischio per la retention.
- **EnvironmentSatisfaction** (4 → 1): in modo controintuitivo, il controfattuale *abbassa* la soddisfazione ambientale. Si tratta di un artefatto dell'ottimizzatore che trova un punto vicino nello spazio delle feature — non una raccomandazione HR significativa. Riflette probabilmente il fatto che nei dati di addestramento alcuni dipendenti con alta soddisfazione ambientale hanno comunque lasciato per altri motivi.
- **WorkLifeBalance** (3 → 1,5): stesso artefatto — l'ottimizzatore si avvicina a un prototipo, non un segnale genuino di retention.
- La variazione del reddito (+€54) è anch'essa trascurabile e artefattuale.

> **Conclusione per HR:** Il modello ha segnalato questo dipendente come ad alto rischio principalmente a causa dell'età (18 anni) e della completa assenza di anzianità — entrambi fattori inevitabili al momento dell'assunzione. Il fatto che sia rimasto nonostante i segnali di allarme suggerisce che il modello manca di feature in grado di catturare i reali motivatori di retention (es. alta soddisfazione ambientale, nessun straordinario). Questo falso positivo è un chiaro esempio di overgeneralizzazione del modello dai segnali demografici e di anzianità a scapito dei dati qualitativi di soddisfazione.

---

## Osservazioni trasversali

| Segnale | Dipendente 214 (VP) | Dipendente 223 (FP) |
|---|---|---|
| Principale driver di rischio | Straordinari + trasferte + bassa anzianità manageriale | Età giovane + anzianità zero |
| Leva più azionabile | Eliminare straordinari, ridurre trasferte | N/A (età e anzianità sono immutabili all'assunzione) |
| Limite del modello evidenziato | Nessuno — il modello era corretto | Sovra-penalizza i profili giovani appena entrati in azienda |
| Variazioni CF artefattuali | Rumore minore sul reddito | EnvironmentSatisfaction ↓, WorkLifeBalance ↓ |

Il confronto mette in luce un limite strutturale di `CounterfactualProto` con feature one-hot e ordinali: l'ottimizzatore può produrre **valori frazionari** per feature binarie (es. `OverTime_Yes = 0,197`) e può spostare i punteggi di soddisfazione in direzioni controintuitive inseguendo un prototipo vicino. Per uno strumento HR in produzione, sarebbe opportuno un post-processing che arrotondi le feature binarie a 0/1 e filtri le diminuzioni di soddisfazione come raccomandazioni HR.